# 🌍 Climate NER — Fine-Tune INDUS on Google Colab

**Model:** `nasa-impact/nasa-smd-ibm-v0.1` (RoBERTa-base, domain-adapted for earth/space science)  
**Dataset:** `ibm-research/Climate-Change-NER` (534 climate paper abstracts, 27 IOB tags)  
**Runtime:** T4 GPU recommended (Runtime → Change runtime type → T4 GPU)  
**Expected training time:** ~12–18 min on T4 for 5 epochs

---
Run cells **top to bottom**. Everything is self-contained.

## 0️⃣  Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️  No GPU — switch runtime to T4 for reasonable training time')

## 1️⃣  Install dependencies

In [ ]:
%%capture
!pip install transformers datasets accelerate seqeval evaluate huggingface-hub -q

## 2️⃣  Clone project and install as a package

In [ ]:
# ── Option A: clone from GitHub (update URL once you push) ───────────────
# !git clone https://github.com/YOUR_USERNAME/CLIMATE-TECH.git
# %cd CLIMATE-TECH

# ── Option B: upload a zip from your machine ─────────────────────────────
# from google.colab import files
# uploaded = files.upload()   # select CLIMATE-TECH.zip
# !unzip CLIMATE-TECH.zip
# %cd CLIMATE-TECH

# ── Option C: mount Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os
# Update this path to wherever you placed the CLIMATE-TECH folder in Drive:
PROJECT_PATH = '/content/drive/MyDrive/CLIMATE-TECH'
os.chdir(PROJECT_PATH)
print('Working directory:', os.getcwd())

In [ ]:
# Install the climate_ner package in editable mode
!pip install -e . -q
print('Package installed ✓')

## 3️⃣  Smoke-test the parser (no GPU needed)

In [ ]:
from climate_ner.dataset import load_climate_ner, LABEL_NAMES

ds = load_climate_ner()
print(f'Train: {len(ds["train"])} sentences')
print(f'Val  : {len(ds["validation"])} sentences')
print(f'Test : {len(ds["test"])} sentences')

# Inspect a sample
ex = ds['train'][11]
print('\nSample tokens :', ex['tokens'][:8])
print('Sample labels :', [LABEL_NAMES[i] for i in ex['ner_tags'][:8]])

# Sanity check
assert len(ds['train']) == 1005, f'Expected 1005, got {len(ds["train"])}'
print('\n✓ Parser smoke-test passed')

## 4️⃣  Configure training

In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────────
OUTPUT_DIR   = '/content/climate-ner-indus'   # where checkpoints are saved
LEARNING_RATE = 2e-5
BATCH_SIZE   = 16     # works on T4; reduce to 8 if you hit OOM
NUM_EPOCHS   = 5

print(f'Output dir  : {OUTPUT_DIR}')
print(f'LR          : {LEARNING_RATE}')
print(f'Batch size  : {BATCH_SIZE}')
print(f'Epochs      : {NUM_EPOCHS}')

## 5️⃣  Run training

In [ ]:
!python -m climate_ner.train \
    --output-dir {OUTPUT_DIR} \
    --lr {LEARNING_RATE} \
    --batch-size {BATCH_SIZE} \
    --epochs {NUM_EPOCHS}

## 6️⃣  Evaluate on test set

In [ ]:
!python -m climate_ner.evaluate --model-dir {OUTPUT_DIR}

## 7️⃣  Run inference on a sample sentence

In [ ]:
from climate_ner.predict import ClimateNER

ner = ClimateNER(OUTPUT_DIR)

sentences = [
    "Intense El Niño rainfall triggered flash floods in lower regions, "
    "damaging maize crops and washing out local bridges.",

    "CO2 emissions from the transport sector accelerated permafrost thaw "
    "and sea-level rise in coastal communities.",

    "CMIP6 models project a 2°C increase in global mean surface temperature "
    "under RCP8.5 by 2100.",
]

for sent in sentences:
    print(f'\nInput: {sent}')
    entities = ner.extract(sent)
    if not entities:
        print('  (no entities detected)')
    else:
        for ent in entities:
            print(f"  [{ent['entity_group']:<30}] '{ent['word']}' ({ent['score']:.3f})")

## 8️⃣  (Optional) Copy model back to Google Drive

In [ ]:
import shutil
DRIVE_MODEL_PATH = '/content/drive/MyDrive/CLIMATE-TECH/models/climate-ner-indus'
shutil.copytree(OUTPUT_DIR, DRIVE_MODEL_PATH, dirs_exist_ok=True)
print(f'Model copied to Drive: {DRIVE_MODEL_PATH}')